In [1]:
import re
from pathlib import Path

import pandas as pd

In [2]:
df = pd.read_csv("../data/processed/adzuna_jobs_cleaned.csv")
df.shape

(1135, 21)

In [3]:
df.dtypes

salary_max       float64
latitude         float64
redirect_url         str
title                str
created              str
salary_min       float64
id                 int64
description          str
longitude        float64
contract_time        str
_search_query        str
company              str
category             str
category_tag         str
location             str
location_area        str
contract_type        str
city                 str
week                 str
day_of_week          str
is_recent           bool
dtype: object

In [6]:
df["created"] = pd.to_datetime(df["created"], errors="coerce")

In [7]:
df["created"].dtype

datetime64[us, UTC]

In [8]:
df_recent = df[df["is_recent"]].copy()
df_recent.shape

(1025, 21)

In [92]:
SKILLS = {
    "Programming": {
        "Python": [r"\bpython\b"],          # ← added \b\b
        "SQL": [r"\bsql\b"],                # ← added \b\b
        "R": [r"\br\b"],
        "Java": [r"\bjava\b"],
        "Scala": [r"\bscala\b"],            # ← FIX: was matching "scalable"
        "JavaScript": [r"\bjavascript\b", r"\bjava script\b"],
        "VBA": [r"\bvba\b"],
        "Pandas": [r"\bpandas\b"],
        "NumPy": [r"\bnumpy\b"],
    },
    "BI & Visualization": {
        "Power BI": [r"\bpower bi\b", r"\bpowerbi\b", r"\bpower-bi\b"],
        "Tableau": [r"\btableau\b"],
        "Qlik": [r"\bqlik\b", r"\bqliksense\b", r"\bqlik sense\b"],
        "Excel": [r"\bexcel\b"],            # ← FIX: was matching "excellent"
        "Looker": [r"\blooker\b"],
        "Dashboards": [r"\bdashboard"],     # voluntarily no closing \b (matches dashboards too)
        "Reporting": [r"\breporting\b"],
        "Data Visualization": [r"\bdata visualization\b", r"\bdata viz\b", r"\bdata visualisation\b"],
    },
    "Cloud & Big Data": {
        "AWS": [r"\baws\b", r"\bamazon web services\b"],   # ← \b avoids "draws", "jaws"
        "Azure": [r"\bazure\b"],
        "GCP": [r"\bgcp\b", r"\bgoogle cloud\b"],
        "Snowflake": [r"\bsnowflake\b"],
        "Databricks": [r"\bdatabricks\b"],
        "Spark": [r"\bspark\b", r"\bpyspark\b"],            # ← \b avoids "sparking"
        "Hadoop": [r"\bhadoop\b"],
    },
    "Data Engineering": {
        "ETL": [r"\betl\b"],                # ← \b just in case
        "Airflow": [r"\bairflow\b"],
        "dbt": [r"\bdbt\b"],
        "Kafka": [r"\bkafka\b"],
        "Data Pipeline": [r"\bdata pipeline\b", r"\bdata pipelines\b"],
    },
    "ML & Stats": {
        "Machine Learning": [r"\bmachine learning\b", r"\bml model\b"],
        "Deep Learning": [r"\bdeep learning\b"],
        "Statistics": [r"\bstatistics\b", r"\bstatistical\b"],
        "NLP": [r"\bnlp\b", r"\bnatural language processing\b"],
        "A/B Testing": [r"\ba/b test", r"\bab testing\b"],
        "Predictive Modeling": [r"\bpredictive model", r"\bforecasting\b"],
    },
}

In [93]:
total_skills = sum(len(cat) for cat in SKILLS.values())
print(f"Total  categorie of skills to extract: {len(SKILLS)}")
print(f"Total skills to extract: {total_skills}")

Total  categorie of skills to extract: 5
Total skills to extract: 35


In [94]:
#function to extract skills from a job description
def extract_skills(description : str) -> list [str]:
    """ Extract canionical skill names from a job descriprion
    args:
        description (str): job description text"""
    ""
     # Defensive: handle missing descriptions 
    if not isinstance(description, str):
        return []
    # 1 - lower case the description for case-insensitive matching
    text = description.lower()
    # 2 - Prepare the result list
    found_skills = []
    # 3 -  iterate through all categories and skills
    for category, skills_dict in SKILLS.items():
        for canonical_name, variants in skills_dict.items():
            for variant in variants:
                # 4 - Use regex to find whole word matches
                if re.search(variant, text):
                    found_skills.append(canonical_name)
                    break  # stop after the first match for this skill
    return found_skills

In [95]:
df_recent["skills"] = df_recent["description"].apply(extract_skills)

In [96]:
df_recent[["title", "skills"]].head(10)

,title,skills
0,Data Analyst,[Dashboards]
1,Data Analyst,"[Python, SQL]"
2,Data Analyst,"[Dashboards, Reporting]"
3,Data Analyst,[]
4,Data Analyst,[Dashboards]
5,Data Analyst,"[Power BI, Excel]"
6,Data Analyst,[]
7,Data Analyst,[Excel]
8,Data Analyst,[]
9,Data Analyst,[]


In [97]:
df_recent["description"].str.len().describe()

count    1025.000000
mean      499.729756
std         5.057496
min       395.000000
25%       500.000000
50%       500.000000
75%       500.000000
max       500.000000
Name: description, dtype: float64

In [98]:
print("Title:", df_recent.loc[1, "title"])
print("---")
print("Description (premiers 500 chars):")
print(df_recent.loc[1, "description"][:500])
print("---")
print("Longueur totale:", len(df_recent.loc[1, "description"]))
print("Skills trouvées:", df_recent.loc[1, "skills"])

Title: Data Analyst
---
Description (premiers 500 chars):
About the job Mercor connects elite creative and technical talent with leading AI research labs. Headquartered in San Francisco, our investors include Benchmark , General Catalyst , Peter Thiel , Adam D'Angelo , Larry Summers , and Jack Dorsey . Position: Artifact Expert — Data Analyst Type: Contract Compensation: $80–$120/hour Location: Remote Role Responsibilities Review existing analytics slides, SQL/Python notebooks, and sheets. Provide detailed, constructive feedback to enhance clarity and…
---
Longueur totale: 500
Skills trouvées: ['Python', 'SQL']


In [99]:
#Title + description
df_recent["text_to_search"] = (
    df_recent["title"].fillna("") + " " + df_recent["description"].fillna("")
)

In [86]:
df_recent["skills"] = df_recent["text_to_search"].apply(extract_skills)

In [100]:
nb_with_skills = (df_recent["skills"].str.len() > 0).sum()
total = len(df_recent)
print(f"Offres avec au moins 1 skill détectée : {nb_with_skills} / {total} ({nb_with_skills/total:.1%})")
print()
print("Stats du nombre de skills par offre :")
print(df_recent["skills"].str.len().describe())

Offres avec au moins 1 skill détectée : 288 / 1025 (28.1%)

Stats du nombre de skills par offre :
count    1025.000000
mean        0.489756
std         1.004940
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         8.000000
Name: skills, dtype: float64


In [101]:
df_recent[["title", "skills"]].head(10)

,title,skills
0,Data Analyst,[Dashboards]
1,Data Analyst,"[Python, SQL]"
2,Data Analyst,"[Dashboards, Reporting]"
3,Data Analyst,[]
4,Data Analyst,[Dashboards]
5,Data Analyst,"[Power BI, Excel]"
6,Data Analyst,[]
7,Data Analyst,[Excel]
8,Data Analyst,[]
9,Data Analyst,[]


In [102]:
empty = df_recent[df_recent["skills"].str.len() == 0].head(5)

for idx, row in empty.iterrows():
    print(f"--- Offre {idx} ---")
    print(f"Title: {row['title']}")
    print(f"Search query: {row['_search_query']}")
    print(f"Description (200 first chars):")
    print(row['description'][:200])
    print()

--- Offre 3 ---
Title: Data Analyst
Search query: data analyst
Description (200 first chars):
Here's your clean, copy-ready job posting: JOB OPENING — Remote Data Analyst ADF Medical Services Inc. We're looking for a sharp, analytical Remote Data Analyst to help us turn data into actionable in

--- Offre 6 ---
Title: Data Analyst
Search query: data analyst
Description (200 first chars):
Job Description Data Analyst Job Type: Contract (Full-Time) Location: Downtown Toronto, ON Work Model: Hybrid (combination of in-office and remote work) Pay Rate: $40 – $45 per hour Overview We are se

--- Offre 8 ---
Title: Data Analyst
Search query: data analyst
Description (200 first chars):
Location: London, ON | Remote | StarTech.com StarTech.com is continuing to grow our Data Analytics team and we are looking for a Data Analyst who is passionate about turning data into actionable insig

--- Offre 9 ---
Title: Data Analyst
Search query: data analyst
Description (200 first chars):
Who We Are Konrad 

In [103]:
                     # ← OK, filtrage de lignes (booléen)
df_skills = df_recent[df_recent["skills"].str.len() > 0].copy()  # ← filtrage de lignes (booléen)
df_skills.shape

(288, 23)

In [104]:
#Explode the skills column to have one skill per row per pair
df_exploded = df_skills.explode("skills")
print(f"Original df_skills: {df_skills.shape}")
print(f"Exploded df_exploed: {df_exploed.shape}")
df_exploed[["title", "skills"]].head(10)

Original df_skills: (288, 23)
Exploded df_exploed: (614, 23)


,title,skills
0,Data Analyst,Dashboards
1,Data Analyst,Python
1,Data Analyst,SQL
2,Data Analyst,Dashboards
2,Data Analyst,Reporting
4,Data Analyst,Dashboards
5,Data Analyst,Power BI
5,Data Analyst,Excel
7,Data Analyst,Excel
10,Data Analyst,Reporting


In [105]:
# Count occurrences of each skill across all postings
top_skills = df_exploded["skills"].value_counts()

print("Top 20 skills in Ontario data postings:")
print(top_skills.head(20))

Top 20 skills in Ontario data postings:
skills
Reporting              93
Machine Learning       43
Python                 41
SQL                    40
Data Pipeline          29
Spark                  24
AWS                    23
Dashboards             20
Power BI               20
Databricks             20
ETL                    20
Azure                  20
Statistics             17
Predictive Modeling    16
Excel                  15
R                      15
Snowflake              11
Java                    6
Pandas                  4
Looker                  3
Name: count, dtype: int64


In [106]:
#Find posting where Scala was matched - inspect context
scala_matches = df_skills[df_skills["skills"].apply(lambda x: "Scala" in x)]
print(f"Number of postings mentioning Scala: {len(scala_matches)}")
print()

for index, row in scala_matches.head(5).iterrows():
     text = row["text_to_search"].lower()
     pos = text.find("scala")
     if pos >=0:
          # Extract 30 chars before and 30 chars after
          start = max(0, pos - 30)
          end = min(len(text), pos + 30)
          print(f"--- Posting {index} ---")
          print(f"Title: {row['title']}")   
          print(f"Context around...{text[start:end]}...")
          print()


Number of postings mentioning Scala: 1

--- Posting 287 ---
Title: Manager/ Sr. Manager - BI Consulting (Life Sciences/Pharma)
Context around...sultant to design and deliver scala…...



In [77]:
##Find posting where Excel was matched - inspect context
excel_matches = df_skills[df_skills["skills"].apply(lambda x: "Excel" in x)]
print(f"Number of postings mentioning Excel: {len(excel_matches)}")
print()

for excel_index, row in excel_matches.head(5).iterrows():
     text = row["text_to_search"].lower()
     pos = text.find("excel")
     if pos >=0:
          # Extract 30 chars before and 30 chars after
          start = max(0, pos - 30)
          end = min(len(text), pos + 30)
          print(f"--- Posting {excel_index} ---")
          print(f"Title: {row['title']}")   
          print(f"Context around...{text[start:end]}...")
          print()

Number of postings mentioning Excel: 35

--- Posting 5 ---
Title: Data Analyst
Context around...th strong technical skills in excel and power bi and demonst...

--- Posting 7 ---
Title: Data Analyst
Context around...rmediaries that allow them to excel in their core businesses...

--- Posting 15 ---
Title: Data Analyst
Context around...t office suite, with advanced excel skills for analysis. pyt...

--- Posting 20 ---
Title: Data Analyst
Context around...formance. the ideal candidate excels at synthesizing informa...

--- Posting 23 ---
Title: Data Analyst
Context around..., and jack dorsey . position: excel sheets contributor compe...



In [108]:
# Build a reverse lookup: skill  name -> category
skill_to_category = {}
for category, skills_dict in SKILLS.items():
    for skill_name in skills_dict.keys():
        skill_to_category[skill_name] = category
#Quick verification
print (f"Total skills mapped: {len(skill_to_category)}")
print()
print("Sample mapping:")
for skill in ["Power BI", "AWS", "Reporting"]:
    print(f"{skill} -> {skill_to_category.get(skill, 'NOT FOUND')}")
    

Total skills mapped: 35

Sample mapping:
Power BI -> BI & Visualization
AWS -> Cloud & Big Data
Reporting -> BI & Visualization


In [109]:
#Map each skill in the exploded df to its category
df_exploded["category"] = df_exploded["skills"].map(skill_to_category)
df_exploded[["title", "skills", "category"]].head(10)

,title,skills,category
0,Data Analyst,Dashboards,BI & Visualization
1,Data Analyst,Python,Programming
1,Data Analyst,SQL,Programming
2,Data Analyst,Dashboards,BI & Visualization
2,Data Analyst,Reporting,BI & Visualization
4,Data Analyst,Dashboards,BI & Visualization
5,Data Analyst,Power BI,BI & Visualization
5,Data Analyst,Excel,BI & Visualization
7,Data Analyst,Excel,BI & Visualization
10,Data Analyst,Reporting,BI & Visualization


In [110]:
#For each category, get the top ski;ss
for category in SKILLS.keys():
    print(f"==={category}===")
    top_in_category = df_exploded[df_exploded["category"] == category]["skills"].value_counts()
    print(top_in_category.head(5))
    print()

===Programming===
skills
Python    41
SQL       40
R         15
Java       6
Pandas     4
Name: count, dtype: int64

===BI & Visualization===
skills
Reporting     93
Dashboards    20
Power BI      20
Excel         15
Looker         3
Name: count, dtype: int64

===Cloud & Big Data===
skills
Spark         24
AWS           23
Databricks    20
Azure         20
Snowflake     11
Name: count, dtype: int64

===Data Engineering===
skills
Data Pipeline    29
ETL              20
dbt               1
Airflow           1
Kafka             1
Name: count, dtype: int64

===ML & Stats===
skills
Machine Learning       43
Statistics             17
Predictive Modeling    16
Deep Learning           3
NLP                     2
Name: count, dtype: int64



In [111]:
# Distribution of postings across the 6 search queries (job categories)
print("Postings per job category (df_skills, 352 postings with skills detected):")
print(df_skills["_search_query"].value_counts())

Postings per job category (df_skills, 352 postings with skills detected):
_search_query
data scientist      73
data analyst        70
data engineer       68
business analyst    33
data consultant     29
BI analyst          15
Name: count, dtype: int64


In [112]:
# For each job category, show the top 10 skills
job_categories = df_exploded["_search_query"].unique()

for job_cat in sorted(job_categories):
    n_postings = df_skills[df_skills["_search_query"] == job_cat].shape[0]
    print(f"=== {job_cat.upper()} (n={n_postings} postings) ===")
    
    top_skills_for_job = (
        df_exploded[df_exploded["_search_query"] == job_cat]["skills"]
        .value_counts()
        .head(10)
    )
    print(top_skills_for_job)
    print()

=== BI ANALYST (n=15 postings) ===
skills
Reporting              8
SQL                    7
Power BI               5
Dashboards             3
AWS                    2
Data Pipeline          2
Azure                  2
R                      1
Predictive Modeling    1
Name: count, dtype: int64

=== BUSINESS ANALYST (n=33 postings) ===
skills
Reporting              24
Predictive Modeling     5
SQL                     3
Statistics              2
Python                  2
Excel                   2
Dashboards              2
Databricks              1
Azure                   1
Tableau                 1
Name: count, dtype: int64

=== DATA ANALYST (n=70 postings) ===
skills
Reporting     31
SQL           14
Excel         13
Python        11
Dashboards    10
Power BI      10
Databricks     9
Spark          9
Snowflake      5
Azure          4
Name: count, dtype: int64

=== DATA CONSULTANT (n=29 postings) ===
skills
Machine Learning    11
Reporting            9
Databricks           2
AWS           

In [115]:
# Reset the index to have unique row labels
df_exploded_reset = df_exploded.reset_index(drop=True)

# Now crosstab will work
pivot = pd.crosstab(
    df_exploded_reset["skills"],
    df_exploded_reset["_search_query"]
)

# Sort by total mentions (most popular skills first)
pivot["total"] = pivot.sum(axis=1)
pivot = pivot.sort_values("total", ascending=False).drop(columns="total")

# Display the top 15 skills
print("Top 15 skills × job category (number of mentions):")
pivot.head(15)

Top 15 skills × job category (number of mentions):


_search_query,BI analyst,business analyst,data analyst,data consultant,data engineer,data scientist
skills,,,,,,
Reporting,8,24,31,9,11,10
Machine Learning,0,0,2,11,5,25
Python,0,2,11,1,15,12
SQL,7,3,14,1,12,3
Data Pipeline,2,0,2,0,23,2
Spark,0,0,9,0,9,6
AWS,2,0,3,2,5,11
Dashboards,3,2,10,0,3,2
Databricks,0,1,9,2,7,1


In [116]:
# Save the enriched dataset (with skills column) for the next session
output_path = Path("../data/processed/adzuna_jobs_with_skills.csv")
df_recent.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}")
print(f"Rows: {len(df_recent)} | Columns: {df_recent.shape[1]} | Size: {output_path.stat().st_size / 1024:.1f} KB")


Saved: ..\data\processed\adzuna_jobs_with_skills.csv
Rows: 1025 | Columns: 23 | Size: 1401.2 KB


In [117]:
# Top 20 skills overall
top_skills_overall = df_exploded["skills"].value_counts().head(20)
top_skills_overall.to_csv("../outputs/top_skills_overall.csv", header=["count"])

# Pivot table by job category
pivot.to_csv("../outputs/skills_by_job_category.csv")

print("Aggregations saved to outputs/")

Aggregations saved to outputs/
